# 20 — Quantum Channels

The last foundation: how a state changes when it is *not* alone. Derive the general quantum evolution — a **channel** — by hand from three things you already own (a unitary, a tensor product, a partial trace), read off its Kraus operators, and see why "decoherence" is entanglement with an environment you then trace out. Then find out what *completely* positive adds, meet the stochastic matrix as the channel that lives on the diagonal, and build the three standard qubit channels on the Bloch ball. Putting two of them on a clock is the work of `01/21`.

**Prerequisites:** `01/06` (unitarity; the stochastic matrix as the lossy cousin of a gate), `01/10` (the rotation $R_y$), `01/13` (gates as $U\rho U^\dagger$; the Bloch vector), `01/14` (purity, $I/2$), `01/15` (trace distance, for exercise 4), `01/17` (the partial trace by blocks), `01/18` (CNOT; the block formula), `01/19` (the partial transpose; the Bell state).

**What you'll be able to do**
- Derive the operator-sum (Kraus) form $\rho \to \sum_k K_k\rho K_k^\dagger$ from "unitary on system + environment, then partial trace", and verify $\sum_k K_k^\dagger K_k = I$ entry by entry.
- Dial how much the environment learns, and watch the qubit's coherence fall while the joint state stays pure.
- Explain *completely* positive, and show a positive map that is not a channel.
- Write a stochastic matrix (a Markov kernel) as a channel.
- Build dephasing, amplitude damping and depolarizing and read their action on the Bloch ball.

What moved the idling qubit of `01/16` off the sphere was not a gate, since nothing reversible shrinks a Bloch vector. Today you meet the object that did it, and you already own all its parts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS
from qot_course.quantum.gates import IDENTITY, CNOT, PAULI_X, ry, is_unitary
from qot_course.quantum.density import density_matrix, purity, bloch_vector, maximally_mixed, density_from_bloch
from qot_course.quantum.composite import tensor, partial_trace, bell_state, apply_channel, depolarizing_channel, is_cptp

rng = np.random.default_rng(0)   # random qubit states for the positivity test of section 2 and the pairs of exercise 4
viz.use_course_style()

## 1. The general evolution, derived: couple, evolve, forget

Start from what is reversible. A closed system evolves by a unitary: $\rho \to U\rho U^\dagger$ (`01/13`). Now let the qubit $S$ interact with something else — an **environment** $E$, which we take to be one more qubit, starting in $|0\rangle_E$. The *pair* is closed, so the pair evolves by a unitary $U$ on $\mathcal H_S \otimes \mathcal H_E$ (`01/17`). We only ever look at $S$ afterwards, so we trace $E$ out (`01/17` §3):

$$
\rho \;\longmapsto\; \rho' = \mathrm{tr}_E\Big[\,U\,\big(\rho \otimes |0\rangle\langle0|_E\big)\,U^\dagger\,\Big].
$$

That is the whole recipe — couple, evolve, forget — and, for a large enough environment, and provided the system starts uncorrelated with it, it is the most general thing that can happen to a quantum system — that converse is Stinespring's theorem (1955), which we quote rather than prove. Let's turn it into a formula you can compute with. Write the partial trace as the sandwich of `01/17`, $\mathrm{tr}_E[X] = \sum_k (I\otimes\langle k|)\,X\,(I\otimes|k\rangle)$, and insert $\rho\otimes|0\rangle\langle0| = (I\otimes|0\rangle)\,\rho\,(I\otimes\langle0|)$:

$$
\rho' = \sum_k \underbrace{(I\otimes\langle k|)\,U\,(I\otimes|0\rangle)}_{K_k}\;\rho\;\underbrace{(I\otimes\langle0|)\,U^\dagger\,(I\otimes|k\rangle)}_{K_k^\dagger}
= \sum_k K_k\,\rho\,K_k^\dagger .
$$

The $K_k$ are $2\times2$ matrices on the system alone — the **Kraus operators** — and entry by entry they are sub-matrices of $U$: $(K_k)_{ss'} = U_{(s,k),\,(s',0)}$ — fix the environment's row index to $k$ and its column index to $0$, keep the system's indices free. The sum $\rho \to \sum_k K_k\rho K_k^\dagger$ is the **operator-sum form** of a **quantum channel**. It has three properties, each inherited from the recipe:

1. **Linear** in $\rho$ (everything in the recipe is).
2. **Trace-preserving**: $\mathrm{tr}\,\rho' = \mathrm{tr}\big(\rho\sum_k K_k^\dagger K_k\big)$, and $\sum_k K_k^\dagger K_k = \sum_k (I\otimes\langle0|)U^\dagger(I\otimes|k\rangle)(I\otimes\langle k|)U(I\otimes|0\rangle) = (I\otimes\langle0|)\,U^\dagger U\,(I\otimes|0\rangle) = I$ — the *completeness relation*, which is unitarity read on the environment's $|0\rangle$ slot, using the **resolution of the identity** $\sum_k|k\rangle\langle k| = I_E$ (checked in the code).
3. **Positive** (each $K\rho K^\dagger$ is), and in fact *completely* positive — section 2. (The helper `is_cptp` checks the completeness relation only: for a Kraus list that is all there is to check, complete positivity being automatic.)

Here is the simplest environment that does something: a CNOT with the system as control and the environment as target (`01/18`). The environment "copies" the bit. Reading off the entries $U_{(s,k),(s',0)}$ of CNOT: $K_0 = |0\rangle\langle0|$ (the environment stays $|0\rangle$ exactly when the system is $|0\rangle$) and $K_1 = |1\rangle\langle1|$ (it flips to $|1\rangle$ exactly when the system is $|1\rangle$). So $\rho' = |0\rangle\langle0|\rho|0\rangle\langle0| + |1\rangle\langle1|\rho|1\rangle\langle1| = \mathrm{diag}(\rho_{00}, \rho_{11})$: the channel **keeps the diagonal and erases the coherence** — because the environment now *knows* the bit, and a bit that has been learned elsewhere can no longer interfere. Applied to $|+\rangle$ it gives $I/2$: the endpoint `01/19` promised, reached by a unitary and a partial trace and nothing else. Let's compute it three ways.

In [ ]:
def kraus_from_dilation(U, dim_env=2):
    # K_k = (I (x) <k|) U (I (x) |0>), environment = RIGHT factor (stride-2 sub-matrices, not contiguous blocks)
    # with environment on the right, rows/cols are indexed (s, e) -> 2s + e, so block (k, 0) is U[e=k rows, e=0 cols]
    ks = []
    for k in range(dim_env):
        ks.append(U.reshape(2, dim_env, 2, dim_env)[:, k, :, 0])
    return ks

def kraus_from_dilation_by_hand(U):
    # the same, entry by entry: (K_k)_{s s'} = U_{(s,k), (s',0)}
    ks = []
    for k in range(2):
        K = np.zeros((2, 2), dtype=complex)
        for s in range(2):
            for s_ in range(2):
                K[s, s_] = U[2 * s + k, 2 * s_ + 0]
        ks.append(K)
    return ks

# environment = right factor, system = control: CNOT (control left, target right) is exactly the course constant
U_copy = CNOT
K = kraus_from_dilation_by_hand(U_copy)
print("is_unitary(U):", is_unitary(U_copy))
for k, Kk in enumerate(K):
    print(f"K_{k} =\n{Kk.real.astype(int)}")
print("by hand == reshape:", all(np.allclose(a, b) for a, b in zip(K, kraus_from_dilation(U_copy))))
# read them literally off U: rows with e = k, columns with e = 0 (stride 2)
print("U (real):\n", U_copy.real.astype(int))
print("K_0 == U[[0,2]][:, [0,2]]:", np.allclose(K[0], U_copy[[0, 2]][:, [0, 2]]), "   K_1 == U[[1,3]][:, [0,2]]:", np.allclose(K[1], U_copy[[1, 3]][:, [0, 2]]))
print("resolution of the identity |0><0| + |1><1| = I:", np.allclose(sum(np.outer(e, e) for e in np.eye(2)), np.eye(2)))
print("completeness sum_k K_k^dagger K_k =\n", sum(Kk.conj().T @ Kk for Kk in K).real.astype(int), "  is_cptp:", is_cptp(K))

rho_plus = density_matrix(KET_PLUS)
# way 1: the recipe itself -- couple, evolve, forget
joint = U_copy @ tensor(rho_plus, density_matrix(KET_0)) @ U_copy.conj().T
via_recipe = partial_trace(joint, keep=[0], dims=[2, 2])
# way 2: the Kraus sum by hand; way 3: the helper
via_kraus = sum(Kk @ rho_plus @ Kk.conj().T for Kk in K)
via_helper = apply_channel(rho_plus, K)
print("\n|+> in:\n", np.round(rho_plus.real, 2))
print("out via recipe:\n", np.round(via_recipe.real, 2))
print("recipe == Kraus sum == apply_channel:", np.allclose(via_recipe, via_kraus), np.allclose(via_kraus, via_helper),
      "   == I/2:", np.allclose(via_recipe, maximally_mixed(2)))
print("purity in / out:", round(purity(rho_plus), 2), "/", round(purity(via_recipe), 2))
print("the joint state after U is the Bell state (pure, entangled):", np.allclose(joint, density_matrix(bell_state())))

**Read the output.** The two Kraus operators read off CNOT's entries are $|0\rangle\langle0|$ and $|1\rangle\langle1|$; their completeness sum is $I$ entry by entry, as unitarity guarantees. Pushing $|+\rangle$ through the recipe — tensor with $|0\rangle_E$, conjugate by CNOT, trace $E$ out — gives $\mathrm{diag}(\tfrac12, \tfrac12) = I/2$, and the Kraus sum and the helper agree to the last digit. Purity $1 \to \tfrac12$. And the last line is the point of the whole module arc: *the joint state after the interaction is the Bell state* — pure and entangled. Nothing was lost; the qubit's coherence became correlation with the environment, and a partial trace cannot see correlation (`01/17` §5). Decoherence is entanglement you are not allowed to look at.

### How much the environment learns

The CNOT copies the bit completely. Let the environment copy it only *partly*: replace the controlled-$X$ by a controlled rotation $R_y(\theta)$ of the environment (`01/10`), so that when the system is $|1\rangle$ the environment turns by $\theta$ instead of flipping. The entries give $K_0 = \mathrm{diag}(1, \cos\tfrac\theta2)$ and $K_1 = \mathrm{diag}(0, \sin\tfrac\theta2)$ (exercise 1 does it by hand); at $\theta = \pi$ they are the copy channel again.

There is a cleaner way to see what survives. Write the joint state after the coupling as $\tfrac{1}{\sqrt2}\big(|0\rangle|e_0\rangle + |1\rangle|e_1\rangle\big)$, where $|e_0\rangle = |0\rangle$ and $|e_1\rangle = R_y(\theta)|0\rangle = \cos\tfrac\theta2|0\rangle + \sin\tfrac\theta2|1\rangle$ are the environment's two *records* of the bit. Tracing the environment out (`01/17`) leaves $\rho'_{01} = \tfrac12\langle e_1|e_0\rangle = \tfrac12\cos\tfrac\theta2$, with the diagonal untouched. **The coherence that survives is the overlap of the two records.** Identical records ($\theta = 0$) mean the environment learned nothing; orthogonal records ($\theta = \pi$) mean the bit can be read off the environment, and nothing is left to interfere. By hand, the qubit's purity is
$$
\mathrm{tr}\,\rho'^2 = \tfrac14 + \tfrac14 + 2\cdot\tfrac14\cos^2\tfrac\theta2 = \tfrac12\big(1 + \cos^2\tfrac\theta2\big),
$$
from $1$ at $\theta = 0$ down to $\tfrac12$ at $\theta = \pi$, while the purity of the pair stays $1$ throughout, because a unitary acted on a pure product. Let's sweep $\theta$ and watch the three numbers.

In [ ]:
def controlled_ry(theta):
    # system = control (left factor), environment = target (right factor): |0><0| (x) I + |1><1| (x) R_y(theta)
    return tensor(np.outer(KET_0, KET_0.conj()), IDENTITY) + tensor(np.outer(KET_1, KET_1.conj()), ry(theta))

theta = 1.1
K_dil = kraus_from_dilation_by_hand(controlled_ry(theta))
print(f"theta = {theta}: K_0 =\n{np.round(K_dil[0].real, 4)}\nK_1 =\n{np.round(K_dil[1].real, 4)}")
print("K_0 == diag(1, cos theta/2):", np.allclose(K_dil[0], np.diag([1, np.cos(theta / 2)])),
      "   K_1 == diag(0, sin theta/2):", np.allclose(K_dil[1], np.diag([0, np.sin(theta / 2)])), "   is_cptp:", is_cptp(K_dil))
print("theta = pi gives the copy channel back:", all(np.allclose(a, b) for a, b in zip(kraus_from_dilation_by_hand(controlled_ry(np.pi)), K)))
e0, e1 = KET_0, ry(theta) @ KET_0                            # the environment's two records of the bit
print(f"overlap of the records <e1|e0> = {np.vdot(e1, e0).real:.4f} = cos(theta/2) = {np.cos(theta / 2):.4f}")

thetas = np.linspace(0, np.pi, 61)
coh, pur_S, pur_SE = [], [], []
for th in thetas:
    U = controlled_ry(th)
    joint_th = U @ tensor(rho_plus, density_matrix(KET_0)) @ U.conj().T
    rho_S = partial_trace(joint_th, keep=[0], dims=[2, 2])
    coh.append(abs(rho_S[0, 1])); pur_S.append(purity(rho_S)); pur_SE.append(purity(joint_th))
coh, pur_S, pur_SE = map(np.array, (coh, pur_S, pur_SE))
print("coherence == cos(theta/2)/2 on the whole sweep:", np.allclose(coh, np.cos(thetas / 2) / 2),
      "   system purity == (1 + cos^2(theta/2))/2:", np.allclose(pur_S, (1 + np.cos(thetas / 2) ** 2) / 2),
      f"   joint purity between {pur_SE.min():.6f} and {pur_SE.max():.6f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
ax.plot(thetas, coh, color=COLORS["quantum"], lw=2.5, label=r"$|\rho'_{01}|$ from couple, evolve, forget")
ax.plot(thetas[::6], np.cos(thetas[::6] / 2) / 2, "o", color=COLORS["quantum"], ms=6, mfc="white", label=r"$\frac{1}{2}\langle e_1|e_0\rangle = \frac{1}{2}\cos\frac{\theta}{2}$ (by hand)")
ax.set_xticks([0, np.pi / 2, np.pi]); ax.set_xticklabels(["0", r"$\pi/2$", r"$\pi$ (CNOT)"])
ax.set_xlabel(r"coupling angle $\theta$"); ax.set_ylabel("coherence of the qubit")
ax.set_title("The coherence that survives\nis the overlap of the two records", pad=12)
ax.legend(fontsize=9, loc="lower left")
ax = axes[1]
ax.plot(thetas, pur_SE, color=COLORS["flow"], lw=2.5, label="purity of the pair (system + environment)")
ax.plot(thetas, pur_S, color=COLORS["highlight"], lw=2.5, label="purity of the qubit alone")
ax.axhline(0.5, color=COLORS["muted"], lw=1.2, ls=":", label=r"$1/2$: the purity of $I/2$")
ax.set_xticks([0, np.pi / 2, np.pi]); ax.set_xticklabels(["0", r"$\pi/2$", r"$\pi$ (CNOT)"])
ax.set_ylim(0.4, 1.05); ax.set_xlabel(r"coupling angle $\theta$"); ax.set_ylabel("purity")
ax.set_title("The pair stays pure;\nthe qubit alone becomes mixed", pad=12)
ax.legend(fontsize=9, loc="lower left")
plt.tight_layout(); plt.show()

**Read the figure.** On the left, the coherence computed by the full recipe (solid line) sits on the by-hand overlap $\tfrac12\cos\tfrac\theta2$ (open circles) over the whole sweep. At $\theta = 1.1$ the records overlap by $0.8525$, and the printed Kraus operators are $\mathrm{diag}(1, 0.8525)$ and $\mathrm{diag}(0, 0.5227)$, exactly $\cos\tfrac\theta2$ and $\sin\tfrac\theta2$. On the right, look at the gap between the curves: the pair stays pure to six decimals at every angle, while the qubit alone falls along $\tfrac12(1 + \cos^2\tfrac\theta2)$ to the floor of $I/2$. Nothing is lost from the pair; what the qubit loses, the environment holds as a record. This partial copy is called **dephasing**, the first standard channel of section 4.

## 2. Completely positive — and a positive map that is not a channel

Any map built from the recipe sends density matrices to density matrices. But the recipe guarantees more than that, and the "more" has a name. Suppose the qubit you push through the channel is *half of an entangled pair* (`01/19`): the channel acts on A, nothing happens to B — the map is $\Lambda \otimes \mathrm{id}$. Physically the output must still be a valid state. A map that stays valid even when applied to one part of a larger system is **completely positive** (CP); a map that only stays valid on its own system is merely *positive*. Channels are CP — in Kraus form, $(\Lambda\otimes\mathrm{id})(\rho_{AB}) = \sum_k (K_k\otimes I)\rho_{AB}(K_k\otimes I)^\dagger$, a sum of sandwiches, hence positive — and, conversely, every CP trace-preserving map has a Kraus form — Choi's theorem (1975; Kraus 1983), quoted here, proved in the references. Hence **CPTP** = channel.

You have already met a map that is positive but *not* completely positive: the **transpose**. On one qubit, $\rho \to \rho^{\mathsf T}$ keeps eigenvalues, hence maps states to states. Applied to half a Bell pair it is `01/19`'s partial transpose, and the output had the eigenvalue $-\tfrac12$ — not a state. So the transpose is not a channel, no environment can implement it, and the very test you used to *certify* entanglement is the standard example of why "positive" is not enough.

The cell below tests both maps on the Bell state, then follows a whole family, $M_s(\rho) = (1-s)\,\rho + s\,\rho^{\mathsf T}$. On one qubit every $M_s$ is positive, since a mixture of two states is a state. On half a Bell pair the output is $(1-s)\,\rho_{AB} + s\,\rho_{AB}^{T_A}$, and you can predict its lowest eigenvalue by hand: $\rho_{AB}^{T_A}$ is half the swap operator (the transpose moves $|00\rangle\langle11|$ to $|10\rangle\langle01|$), which has eigenvalue $-\tfrac12$ on the singlet $(|01\rangle - |10\rangle)/\sqrt2$, where the Bell state has no weight; so the output has eigenvalue $-s/2$. Beside it runs a genuine channel, depolarizing on half the pair (section 4 builds it: with probability $p$, replace A by $I/2$), with output $(1-p)\,\rho_{AB} + p\,I/4$ (exercise 2) and lowest eigenvalue $p/4$.

In [ ]:
rho_bell = density_matrix(bell_state())

def apply_on_A(kraus, rho4):
    return sum(tensor(Kk, IDENTITY) @ rho4 @ tensor(Kk, IDENTITY).conj().T for Kk in kraus)

def partial_transpose_A(rho4):      # transpose the A-indices: (rho^{T_A})_{ab,a'b'} = rho_{a'b,ab'}
    return rho4.reshape(2, 2, 2, 2).transpose(2, 1, 0, 3).reshape(4, 4)

out_dephase = apply_on_A(K, rho_bell)                      # the copy channel on half a Bell pair
out_transpose = partial_transpose_A(rho_bell)               # the transpose map on half a Bell pair
for name, m in [("copy channel (x) id on Bell", out_dephase), ("transpose (x) id on Bell", out_transpose)]:
    ev = np.linalg.eigvalsh(m).real
    ev = np.where(np.abs(ev) < 1e-9, 0.0, ev)
    print(f"{name:32s} eigenvalues {np.round(ev, 3)}   valid state: {ev.min() >= -1e-9}")
print("copy (x) id on Bell == the correlated coins of 01/17:",
      np.allclose(out_dephase, 0.5 * density_matrix(tensor(KET_0, KET_0)) + 0.5 * density_matrix(tensor(KET_1, KET_1))))
SWAP = np.eye(4)[[0, 2, 1, 3]]
print("Bell^{T_A} == SWAP/2:", np.allclose(out_transpose, SWAP / 2))

# the family M_s = (1 - s) id + s transpose, tested on one qubit and on half a Bell pair
s_grid = np.linspace(0, 1, 41)
pure_states = [density_from_bloch(v / np.linalg.norm(v)) for v in rng.normal(size=(500, 3))]   # 500 random pure states
one_qubit = np.array([min(np.linalg.eigvalsh((1 - s) * r + s * r.T).min() for r in pure_states) for s in s_grid])
half_bell = np.array([np.linalg.eigvalsh((1 - s) * rho_bell + s * out_transpose).min() for s in s_grid])
depol_bell = np.array([np.linalg.eigvalsh(apply_on_A(depolarizing_channel(s), rho_bell)).min() for s in s_grid])
print(f"\nM_s on 500 random pure states of one qubit: lowest eigenvalue over all s = {one_qubit.min():+.2e}   >= -1e-12: {one_qubit.min() >= -1e-12}")
print("M_s on half a Bell pair: lowest eigenvalue == -s/2:", np.allclose(half_bell, -s_grid / 2),
      f"  (at s = 0.1: {half_bell[4]:+.3f})")
print("depolarizing(p) on half a Bell pair: lowest eigenvalue == p/4:", np.allclose(depol_bell, s_grid / 4))

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.axhline(0, color=COLORS["text"], lw=0.8)
ax.plot(s_grid, one_qubit, color=COLORS["quantum"], lw=2.5, label=r"$M_s$ on one qubit (lowest over 500 states)")
ax.plot(s_grid, half_bell, color=COLORS["negative"], lw=2.5, label=r"$M_s$ on half a Bell pair")
ax.plot(s_grid[::4], -s_grid[::4] / 2, "o", color=COLORS["negative"], ms=6, mfc="white", label=r"$-s/2$ (by hand)")
ax.plot(s_grid, depol_bell, color=COLORS["flow"], lw=2.5, label=r"depolarizing($p$) on half a Bell pair")
ax.plot(s_grid[::4], s_grid[::4] / 4, "s", color=COLORS["flow"], ms=6, mfc="white", label=r"$p/4$ (by hand)")
ax.set_xlabel(r"strength $s$ (or $p$)"); ax.set_ylabel("lowest eigenvalue of the output")
ax.set_title("Positive on one qubit is not enough:\nthe test on half a Bell pair", pad=12)
ax.legend(fontsize=9, loc="lower left")
plt.tight_layout(); plt.show()

**Read the output and the figure.** The copy channel applied to half a Bell pair gives eigenvalues $(\tfrac12, \tfrac12, 0, 0)$ — a valid state, and in fact exactly the correlated coins of `01/17` §5: dephasing one half of a Bell pair turns quantum correlation into a shared classical coin. The transpose applied to the same half gives $-\tfrac12$: not a state, so not a channel. Same input, same "positive on one qubit" credentials, opposite verdicts — that is what *completely* adds.

In the figure, the blue line (the lowest eigenvalue of $M_s(\rho)$ over $500$ random pure states of one qubit) never drops below zero: on its own system every $M_s$ looks harmless. The coral line, the same map on half a Bell pair, is negative as soon as $s > 0$ and follows $-s/2$ exactly ($-0.050$ at $s = 0.1$): even a tenth of a transpose is unphysical, and no environment can implement it. The green line, depolarizing on half the pair, stays at $p/4 \ge 0$, as a channel's must.

## 3. The classical case: a stochastic matrix is a channel

A **stochastic matrix** $T$ is a square matrix whose columns are probability vectors: non-negative entries, each column summing to $1$. It is also called a **Markov kernel**, because it is the one-step rule of a Markov chain: $T_{ij} = P(\text{next} = i \mid \text{now} = j)$, and one step later the distribution is $p' = Tp$. It keeps probabilities valid: $Tp$ is non-negative, and column $j$ contributes $p_j$ times $1$ to its total. With
$$
T = \begin{pmatrix} 0.9 & 0.3 \\ 0.1 & 0.7 \end{pmatrix}, \qquad p = (0.6,\ 0.4),
$$
you get by hand $p' = (0.9\cdot0.6 + 0.3\cdot0.4,\ 0.1\cdot0.6 + 0.7\cdot0.4) = (0.66,\ 0.34)$. Repeated, the chain settles on the **stationary distribution** $\pi$ with $T\pi = \pi$: $0.1\,\pi_0 = 0.3\,\pi_1$ and $\pi_0 + \pi_1 = 1$ give $\pi = (0.75,\ 0.25)$.

The quantum packaging uses one Kraus operator per entry, $K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$. By hand, $K_{ij}\,\rho\,K_{ij}^\dagger = T_{ij}\,\rho_{jj}\,|i\rangle\langle i|$, so the output is diagonal with entries $\sum_j T_{ij}\rho_{jj} = (Tp)_i$, where $p$ is the diagonal of $\rho$; and completeness is the column sums, $\sum_{ij} K_{ij}^\dagger K_{ij} = \sum_{j}\big(\sum_i T_{ij}\big)|j\rangle\langle j| = I$. This channel maps $\mathrm{diag}(p)$ to $\mathrm{diag}(Tp)$ and erases every coherence. Classical Markov evolution is the channel that lives on the diagonal; that is how the dictionary row "stochastic matrix ↔ channel" is earned.

The reversible case is a permutation, such as the bit flip, which moves probability without mixing it; its quantum cousin is the unitary, the channel with one Kraus operator $K = U$. The bit flip shows how much more a channel carries than its diagonal. As a classical channel (Kraus operators $|1\rangle\langle0|$ and $|0\rangle\langle1|$) it swaps the populations and kills the coherence of $|+\rangle$; the unitary $X$ swaps the same populations and keeps the coherence. Let's check every claim, run the chain for ten steps, and follow the coherence of $|+\rangle$ under three channels.

In [ ]:
T = np.array([[0.9, 0.3],
              [0.1, 0.7]])                                  # columns sum to 1: T[i, j] = P(next = i | now = j)
K_T = [np.sqrt(T[i, j]) * np.outer(np.eye(2)[i], np.eye(2)[j]) for i in range(2) for j in range(2)]
print("stochastic channel: is_cptp:", is_cptp(K_T), "   column sums:", T.sum(axis=0))
p = np.array([0.6, 0.4])
print("T p            =", T @ p)
print("channel(diag p)=", np.round(np.diag(apply_channel(np.diag(p).astype(complex), K_T)).real, 4))
print("on |+> (coherence erased, diagonal moved):\n", np.round(apply_channel(rho_plus, K_T).real, 3))

# ten steps of the chain from p0 = (0, 1), classically and as a channel
n_steps = np.arange(11)
p_n, rho_n = [np.array([0.0, 1.0])], [np.diag([0.0, 1.0]).astype(complex)]
for _ in n_steps[1:]:
    p_n.append(T @ p_n[-1]); rho_n.append(apply_channel(rho_n[-1], K_T))
p_n = np.array(p_n)
print("\ndiagonal of channel^n(diag p0) == T^n p0 for n = 0..10:", all(np.allclose(np.diag(r).real, q) for r, q in zip(rho_n, p_n)))
w, v = np.linalg.eig(T)
pi = np.real(v[:, np.argmin(np.abs(w - 1))]); pi = pi / pi.sum()
print("stationary distribution pi =", np.round(pi, 4), "   after 10 steps:", np.round(p_n[-1], 4))

# the bit flip, two ways: as a classical (stochastic) channel and as the unitary X
P_flip = np.array([[0.0, 1.0], [1.0, 0.0]])
K_flip = [np.sqrt(P_flip[i, j]) * np.outer(np.eye(2)[i], np.eye(2)[j]) for i in range(2) for j in range(2) if P_flip[i, j] > 0]
for name, ks in [("bit flip as a stochastic channel", K_flip), ("bit flip as the unitary X", [PAULI_X])]:
    out = apply_channel(rho_plus, ks)
    print(f"{name:34s} on diag(0.6, 0.4) -> diag {np.round(np.diag(apply_channel(np.diag(p).astype(complex), ks)).real, 2)}   coherence of |+> after: {abs(out[0, 1]):.2f}")

# the coherence of |+> along repeated steps of three channels
coh_T, coh_D, coh_X = [], [], []
r_T = r_D = r_X = rho_plus
for _ in n_steps:
    coh_T.append(abs(r_T[0, 1])); coh_D.append(abs(r_D[0, 1])); coh_X.append(abs(r_X[0, 1]))
    r_T, r_D, r_X = apply_channel(r_T, K_T), apply_channel(r_D, K_dil), apply_channel(r_X, [PAULI_X])
print(f"partial copy (theta = {theta}): coherence after 1, 2, 3 steps:", np.round(coh_D[1:4], 4), f" (ratio cos(theta/2) = {np.cos(theta / 2):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ax = axes[0]
ax.plot(n_steps, p_n[:, 0], "o-", color=COLORS["source"], lw=2, ms=6, label=r"$p_n(0)$: chain, $T^n p_0$")
ax.plot(n_steps, p_n[:, 1], "s-", color=COLORS["target"], lw=2, ms=6, label=r"$p_n(1)$")
ax.plot(n_steps, [np.diag(r).real[0] for r in rho_n], "x", color=COLORS["text"], ms=8, label="diagonal of the channel output")
ax.axhline(pi[0], color=COLORS["source"], lw=1.2, ls=":"); ax.axhline(pi[1], color=COLORS["target"], lw=1.2, ls=":")
ax.set_xlabel("step $n$"); ax.set_ylabel("probability"); ax.set_ylim(-0.05, 1.05)
ax.set_title("A Markov chain and its channel:\nthe same diagonal, step after step", pad=12)
ax.legend(fontsize=9, loc="center right", bbox_to_anchor=(1.0, 0.42))
ax = axes[1]
ax.plot(n_steps, coh_X, "o-", color=COLORS["quantum"], lw=2, ms=6, label=r"unitary $X$ (one Kraus operator)")
ax.plot(n_steps, coh_D, "s-", color=COLORS["flow"], lw=2, ms=6, label=r"partial copy, $\theta = 1.1$ (section 1)")
ax.plot(n_steps, coh_T, "^-", color=COLORS["highlight"], lw=2, ms=7, label=r"stochastic channel of $T$")
ax.set_xlabel("step $n$"); ax.set_ylabel(r"coherence $|\rho_{01}|$ of $|+\rangle$"); ax.set_ylim(-0.03, 0.55)
ax.set_title("Off the diagonal, the three channels part ways", pad=12)
ax.legend(fontsize=9, loc="center right")
plt.tight_layout(); plt.show()

**Read the output and the figure.** The stochastic channel passes `is_cptp` because each column of $T$ sums to $1$, and its output diagonal is the by-hand $Tp = (0.66, 0.34)$. On the left, the chain started at $p_0 = (0, 1)$ moves toward the stationary $(0.75, 0.25)$ (dotted lines), nearly reached after ten steps ($0.7455$), and the black crosses, the diagonal of the channel's output, sit on the chain at every step. On the right is what the diagonal cannot show. The unitary $X$ keeps the coherence of $|+\rangle$ at $\tfrac12$; the partial copy of section 1 multiplies it by $\cos\tfrac\theta2 = 0.8525$ at each step; the stochastic channel wipes it out in one step. The bit flip says the same in one line: both versions send $\mathrm{diag}(0.6, 0.4)$ to $\mathrm{diag}(0.4, 0.6)$, but only $X$ keeps the coherence ($0.50$ against $0.00$). The off-diagonal is where the quantum part of a channel begins.

## 4. The standard qubit channels

Three channels come back again and again, in noise models and in every textbook (Nielsen & Chuang). Each is a short list of Kraus operators, and each has a picture on the Bloch ball of `01/13` that you can work out from the entries with $x = 2\,\mathrm{Re}\,\rho_{01}$, $y = -2\,\mathrm{Im}\,\rho_{01}$, $z = \rho_{00} - \rho_{11}$.

**Dephasing** (phase damping) is the partial copy of section 1, written with the surviving fraction $\sqrt{1-\lambda}$ in place of $\cos\tfrac\theta2$: $K_0 = \mathrm{diag}(1, \sqrt{1-\lambda})$, $K_1 = \mathrm{diag}(0, \sqrt\lambda)$. It leaves the diagonal alone and multiplies $\rho_{01}$ by $\sqrt{1-\lambda}$; on the ball, $(x, y, z) \to (\sqrt{1-\lambda}\,x,\ \sqrt{1-\lambda}\,y,\ z)$, a squeeze toward the $z$ axis.

**Amplitude damping**: the excited state leaks energy into the environment — $|1\rangle_S|0\rangle_E \to \sqrt{1-\gamma}\,|1\rangle|0\rangle + \sqrt\gamma\,|0\rangle|1\rangle$, with $|0\rangle|0\rangle$ untouched, a partial swap (a rotation that hands the excitation from the system to the environment) whose Kraus operators the cell reads with the function of section 1 — giving $K_0 = \mathrm{diag}(1, \sqrt{1-\gamma})$ and $K_1 = \sqrt\gamma\,|0\rangle\langle1|$. Entry by entry: $\rho_{11} \to (1-\gamma)\rho_{11}$ (population relaxes to $|0\rangle$), $\rho_{01} \to \sqrt{1-\gamma}\,\rho_{01}$ (coherence shrinks by the *square root*). On the ball, $z = 1 - 2\rho_{11}$ becomes $\gamma + (1-\gamma)\,z$, so $(x, y, z) \to (\sqrt{1-\gamma}\,x,\ \sqrt{1-\gamma}\,y,\ \gamma + (1-\gamma)z)$: the ball shrinks and slides up toward $|0\rangle$, the one state that stays put.

**Depolarizing**: with probability $p$ replace the state by $I/2$ — $\rho \to (1-p)\rho + p\,I/2$, Kraus $\sqrt{1 - 3p/4}\,I$ and $\sqrt{p/4}\,X, Y, Z$ (exercise 3 checks the identity $\tfrac14(\rho + X\rho X + Y\rho Y + Z\rho Z) = I/2$). On the ball, $\vec r \to (1-p)\,\vec r$, and the purity of a pure input becomes $\tfrac12\big(1 + (1-p)^2\big)$.

Let's build the three, check each claim on a tilted state with a complex coherence, and draw the image of the ball's $x$–$z$ cross-section.

In [ ]:
def dephasing(lmbda):
    return [np.diag([1, np.sqrt(1 - lmbda)]).astype(complex), np.diag([0, np.sqrt(lmbda)]).astype(complex)]

def amplitude_damping(gamma):
    return [np.diag([1, np.sqrt(1 - gamma)]).astype(complex), np.sqrt(gamma) * np.outer(KET_0, KET_1.conj())]

lam, gam, pdep = 0.36, 0.25, 0.3
print("partial copy at theta = 1.1 == dephasing(lambda = sin^2(theta/2)):",
      all(np.allclose(a, b) for a, b in zip(K_dil, dephasing(np.sin(theta / 2) ** 2))))

# amplitude damping from a dilation: rotate |10> toward |01> (index 2s + e: |10> -> 2, |01> -> 1)
U_ad = np.eye(4, dtype=complex)
U_ad[2, 2], U_ad[1, 2], U_ad[2, 1], U_ad[1, 1] = np.sqrt(1 - gam), np.sqrt(gam), -np.sqrt(gam), np.sqrt(1 - gam)
K_ad = kraus_from_dilation_by_hand(U_ad)
print("partial swap is unitary:", is_unitary(U_ad), "   its Kraus operators == amplitude_damping(0.25):",
      all(np.allclose(a, b) for a, b in zip(K_ad, amplitude_damping(gam))))

rho_t = density_matrix(np.array([np.cos(0.35), np.exp(0.4j) * np.sin(0.35)]))    # a tilted state with a complex coherence
channels = {"dephasing(0.36)": dephasing(lam), "amplitude_damping(0.25)": amplitude_damping(gam), "depolarizing(0.3)": depolarizing_channel(pdep)}
for name, ks in channels.items():
    out = apply_channel(rho_t, ks)
    print(f"\n{name:24s} is_cptp: {is_cptp(ks)}")
    print(f"  rho_11: {rho_t[1,1].real:.4f} -> {out[1,1].real:.4f}   rho_01: {rho_t[0,1]:.4f} -> {out[0,1]:.4f}")
print("\nchecks against the entry-by-entry claims:")
out = apply_channel(rho_t, dephasing(lam));         print("  dephasing:  rho_01 * sqrt(1-lambda) ==", np.isclose(out[0, 1], rho_t[0, 1] * np.sqrt(1 - lam)), "  diagonal untouched:", np.isclose(out[1, 1], rho_t[1, 1]))
out = apply_channel(rho_t, amplitude_damping(gam)); print("  amp. damp.: rho_11 * (1-gamma) ==", np.isclose(out[1, 1], rho_t[1, 1] * (1 - gam)), "  rho_01 * sqrt(1-gamma) ==", np.isclose(out[0, 1], rho_t[0, 1] * np.sqrt(1 - gam)))
out = apply_channel(rho_t, depolarizing_channel(pdep)); print("  depolarizing: (1-p) rho + p I/2 ==", np.allclose(out, (1 - pdep) * rho_t + pdep * np.eye(2) / 2), "  purity (1+(1-p)^2)/2 ==", np.isclose(purity(out), (1 + (1 - pdep) ** 2) / 2))

# the Bloch-ball maps, read off by hand, against the channels
bloch_by_hand = {"dephasing(0.36)": lambda r: np.array([np.sqrt(1 - lam) * r[0], np.sqrt(1 - lam) * r[1], r[2]]),
                 "amplitude_damping(0.25)": lambda r: np.array([np.sqrt(1 - gam) * r[0], np.sqrt(1 - gam) * r[1], gam + (1 - gam) * r[2]]),
                 "depolarizing(0.3)": lambda r: (1 - pdep) * r}
test_vectors = [v * rng.uniform() ** (1 / 3) / np.linalg.norm(v) for v in rng.normal(size=(200, 3))]   # 200 points in the ball
for name, ks in channels.items():
    ok = all(np.allclose(bloch_vector(apply_channel(density_from_bloch(v), ks)), bloch_by_hand[name](v)) for v in test_vectors)
    print(f"  Bloch map of {name:24s} == the by-hand formula on 200 points: {ok}")
for label, v in [("|0>", [0, 0, 1]), ("|1>", [0, 0, -1]), ("|+>", [1, 0, 0])]:
    print(f"  amplitude_damping(0.25) sends {label} to Bloch vector {np.round(bloch_vector(apply_channel(density_from_bloch(np.array(v, float)), amplitude_damping(gam))), 3)}")
print(f"  dephasing(0.36) sends |+> to Bloch vector {np.round(bloch_vector(apply_channel(density_from_bloch(np.array([1.0, 0, 0])), dephasing(lam))), 3)}")
print(f"  depolarizing(0.3) multiplies every Bloch vector by 1 - p = {1 - pdep:.1f}")

angles = np.linspace(0, 2 * np.pi, 241)
circle = np.stack([np.sin(angles), np.zeros_like(angles), np.cos(angles)], axis=1)     # the x-z great circle
marks = {r"$|0\rangle$": [0, 0, 1], r"$|1\rangle$": [0, 0, -1], r"$|+\rangle$": [1, 0, 0], r"$|-\rangle$": [-1, 0, 0]}
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.6))
for ax, (name, ks), col in zip(axes, channels.items(), [COLORS["flow"], COLORS["highlight"], COLORS["source"]]):
    img = np.array([bloch_vector(apply_channel(density_from_bloch(v), ks)) for v in circle])
    ax.plot(circle[:, 0], circle[:, 2], color=COLORS["muted"], lw=1.5, ls="--", label="the sphere (pure states)")
    ax.fill(img[:, 0], img[:, 2], color=col, alpha=0.35, lw=0)
    ax.plot(img[:, 0], img[:, 2], color=col, lw=2.5, label="its image")
    for lab, v in marks.items():
        w = bloch_vector(apply_channel(density_from_bloch(np.array(v, float)), ks))
        ax.annotate("", xy=(w[0], w[2]), xytext=(v[0], v[2]), arrowprops=dict(arrowstyle="->", color=COLORS["text"], lw=1.2))
        ax.text(1.18 * v[0] - 0.08, 1.16 * v[2] - 0.04, lab, fontsize=10, color=COLORS["text"])
    ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$z$"); ax.set_title(name, pad=10)
fig.suptitle("What each channel does to the Bloch ball (the x–z cross-section)", fontsize=14, fontweight="bold")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=2, fontsize=9)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

**Read the output and the figure.** The partial copy at $\theta = 1.1$ is exactly `dephasing` with $\lambda = \sin^2\tfrac\theta2$, and the partial swap hands back exactly the Kraus operators of amplitude damping: both come from "couple, evolve, forget". All three channels pass `is_cptp`, and every entry-by-entry claim and every by-hand Bloch map (on $200$ random points of the ball) prints `True`. In the figure, the dashed circle is the sphere of pure states and the coloured region is its image. Dephasing (left) keeps the poles and pulls $|\pm\rangle$ in to $x = \pm0.8$. Amplitude damping (centre) keeps $|0\rangle$, lifts $|1\rangle$ to $z = -0.5$ and sends $|+\rangle$ to $(0.866, 0, 0.25)$, up and in. Depolarizing (right) shrinks everything toward the centre by the factor $0.7$. Three shapes, all inside the ball, as the image of a channel must be. `01/21` puts the first two on a clock.

## Your turn

1. **Dephasing, derived.** Write the $4\times4$ of the controlled-$R_y(\theta)$ of section 1 (system = control, environment = target) by the block formula of `01/17`, read off $(K_k)_{ss'} = U_{(s,k),(s',0)}$ by hand, and show $K_0 = \mathrm{diag}(1, \cos\tfrac\theta2)$, $K_1 = \mathrm{diag}(0, \sin\tfrac\theta2)$, with $K_0^\dagger K_0 + K_1^\dagger K_1 = I$ entry by entry. Then show that dephasing is also *measure $Z$ and forget the result*, $\rho \to (1 - q)\rho + q\,Z\rho Z$, for the right $q(\lambda)$: two Kraus sets, one channel.
2. **Noise on half a Bell pair, the right meter.** Push one qubit of the Bell state through `depolarizing_channel(p)` for $p \in [0, 1]$ and plot the most negative eigenvalue of the partial transpose (`01/19`) against $p$. Show by hand that the output is the Werner state of `01/19` exercise 4 with weight $1 - p$, so entanglement ends at $p = 2/3$. Then plot `entanglement_entropy` of the output: it stays at $1$ bit, because $\rho_B = I/2$ whatever happens to A. Say in one sentence why that curve tells you nothing here (`01/19` §3: the global state is no longer pure).
3. **Paulis.** Prove by hand that $\tfrac14\big(\rho + X\rho X + Y\rho Y + Z\rho Z\big) = \tfrac{I}{2}$ for every qubit $\rho$ (write $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ and use $XYX = -Y$ and its cousins), hence that the Kraus set of section 4 implements $(1-p)\rho + pI/2$. Then the **bit flip** $\{\sqrt{1-p}\,I, \sqrt p\,X\}$: show that it leaves $|\pm\rangle$ untouched and shrinks the $y$ and $z$ components of the Bloch vector by $1 - 2p$.
4. **Harder: what a channel cannot do.** Prove from the Kraus form that a channel *contracts* the trace distance (`01/15`): $T(\Lambda\rho, \Lambda\sigma) \le T(\rho, \sigma)$. Hint: write $\rho - \sigma = P - Q$ with $P, Q \ge 0$ from its eigen-decomposition, so $T = \mathrm{tr}P = \mathrm{tr}Q$; then use the triangle inequality for $\|\cdot\|_1$, $\|A\|_1 = \mathrm{tr}A$ for $A \ge 0$, and $\mathrm{tr}\,\Lambda P = \mathrm{tr}P$. Check it numerically on random pairs of qubit states for the three channels of section 4. Finally, argue in two sentences why this contraction is the reason no measurement downstream of noise can recover what the noise erased, the honest accounting behind the hardware floor of Module 05.

## What you built

- You can derive the operator-sum form from "couple, evolve, forget", read the Kraus operators off the entries of $U$, and recognise $\sum_k K_k^\dagger K_k = I$ as unitarity read on the environment's $|0\rangle$ slot.
- You can show that decoherence is entanglement you then trace out: the surviving coherence is the overlap of the environment's two records.
- You can say what *completely* positive adds: the transpose, and even a tenth of it, fails the half-Bell-pair test that every channel passes.
- You can write a stochastic matrix (Markov kernel) $p \to Tp$ as the channel with Kraus operators $\sqrt{T_{ij}}\,|i\rangle\langle j|$.
- You can build dephasing, amplitude damping and depolarizing from their Kraus operators and draw what each does to the Bloch ball.

That is the vocabulary of open quantum systems, earned from a unitary, a tensor product and a partial trace. Well done.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| stochastic matrix (Markov kernel) $p \to Tp$ | quantum channel $\rho \to \sum_k K_k\rho K_k^\dagger$, $\sum_k K_k^\dagger K_k = I$ |

## What's next

The channels you built today have a strength — $\lambda$, $\gamma$, $p$ — but no clock. A real qubit's noise grows with time: `01/16` let a qubit idle and saw its coherence fall as $e^{-t/T_2}$ while its $\langle Z\rangle$, started from $|+\rangle$, rose toward $1$ (the qubit relaxing to $|0\rangle$) as $1 - e^{-t/T_1}$. In `01/21_t1_and_t2_channels` you put amplitude damping and dephasing on a clock, derive why both laws must be exponentials, compose the two channels into one $T_2$, rebuild from the calibration snapshot the channel behind the idle data of `01/16`, and prove why $T_2 \le 2T_1$. That closes Module 01.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, Cambridge University Press (2010) — the operator-sum representation, environment models, the unitary freedom of Kraus operators, the standard qubit channels and the contractivity of the trace distance. doi:10.1017/CBO9780511976667
- W. F. Stinespring, *Positive functions on C*-algebras*, Proc. Amer. Math. Soc. **6**, 211 (1955) — every channel is "unitary on a larger system, then a partial trace". doi:10.1090/S0002-9939-1955-0069403-4
- M.-D. Choi, *Completely positive linear maps on complex matrices*, Linear Algebra Appl. **10**, 285 (1975) — complete positivity and the Kraus form. doi:10.1016/0024-3795(75)90075-0
- K. Kraus, *States, Effects, and Operations*, Lecture Notes in Physics, Springer (1983). doi:10.1007/3-540-12732-1
- M. M. Wilde, *Quantum Information Theory*, 2nd ed., Cambridge University Press (2017) — noisy evolution and the Choi–Kraus theorem. doi:10.1017/9781316809976

**Previous:** `notebooks/01_QuantumFoundations/19_entanglement.ipynb` · **Next:** `notebooks/01_QuantumFoundations/21_t1_and_t2_channels.ipynb`